# 29. 年別CSVからのデータ再構築
出典：FX (3).ipynb、元セルindex [63, 64, 65]。保存出力は results/imported_fx3/。
研究履歴の原本です。Notebookの変数・価格CSV・学習済みファイルに依存します。
失敗した試行も保管しています。一括実行やAPI接続を開始する入口ではありません。
元コード内の指示・自動判定名は資料として保存しています。独立した検証済みの結論とは区別してください。


## 元セルindex 63
構文状態：valid


In [ ]:
# ============================================================
# USDJPY ORIGINAL CSV SOURCE AUDIT
#
# 目的:
#   1. 元USDJPY CSVを自動探索
#   2. CSVを1ファイルずつ独立監査
#   3. >5% gap / 即時逆転 / 異常日を特定
#   4. どのCSVがraw contaminationの原因か特定
#   5. CSV間の境界jumpも確認
#
# 重要:
#   ・データは削除しない
#   ・barsは上書きしない
#   ・異常値を閾値で勝手に除去しない
# ============================================================

from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 300)


# ============================================================
# 0. 設定
# ============================================================

GAP_THRESHOLD = 0.05
# 連続15分足間で5%以上飛んだら監査対象

EXTREME_2_THRESHOLD = 0.02

OUTPUT_DIR = Path("csv_source_audit")
OUTPUT_DIR.mkdir(exist_ok=True)

MAX_FILES = 500


# ============================================================
# 1. 表示関数
# ============================================================

def show_table(title, df, n=100):

    print()
    print("=" * 120)
    print(title)
    print("=" * 120)

    if df is None or len(df) == 0:

        print("No data")
        return

    try:

        display(
            df.head(n)
        )

    except Exception:

        print(
            df.head(n).to_string()
        )


# ============================================================
# 2. CSV読込関数
#
# comma / semicolon / tab をある程度自動対応
# ============================================================

def read_csv_flexible(path):

    attempts = []

    # ----------------------------
    # 通常comma
    # ----------------------------

    try:

        df = pd.read_csv(
            path,
            low_memory=False
        )

        attempts.append(df)

    except Exception:

        pass


    # ----------------------------
    # separator自動推定
    # ----------------------------

    try:

        df = pd.read_csv(
            path,
            sep=None,
            engine="python"
        )

        attempts.append(df)

    except Exception:

        pass


    # ----------------------------
    # ; 区切り
    # ----------------------------

    try:

        df = pd.read_csv(
            path,
            sep=";",
            low_memory=False
        )

        attempts.append(df)

    except Exception:

        pass


    # ----------------------------
    # tab
    # ----------------------------

    try:

        df = pd.read_csv(
            path,
            sep="\t",
            low_memory=False
        )

        attempts.append(df)

    except Exception:

        pass


    if not attempts:

        raise RuntimeError(
            "CSVを読み込めませんでした。"
        )


    # 最も列数の多い候補を採用
    attempts.sort(
        key=lambda x: x.shape[1],
        reverse=True
    )

    return attempts[0]


# ============================================================
# 3. OHLC正規化
# ============================================================

def normalize_csv_ohlc(df, path):

    x = df.copy()

    # ----------------------------
    # column names
    # ----------------------------

    x.columns = [
        str(c).strip().lower()
        for c in x.columns
    ]


    # よくある別名を吸収

    aliases = {

        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",

        "bidopen": "open",
        "bidhigh": "high",
        "bidlow": "low",
        "bidclose": "close",

        "open_bid": "open",
        "high_bid": "high",
        "low_bid": "low",
        "close_bid": "close",

        "<open>": "open",
        "<high>": "high",
        "<low>": "low",
        "<close>": "close",

        "<date>": "date",
        "<time>": "time",
    }


    x = x.rename(
        columns={
            c: aliases.get(c, c)
            for c in x.columns
        }
    )


    # ========================================================
    # Timestamp
    # ========================================================

    if isinstance(
        x.index,
        pd.DatetimeIndex
    ):

        idx = pd.to_datetime(
            x.index,
            utc=True,
            errors="coerce"
        )


    else:

        # --------------------------------
        # timestamp単独列
        # --------------------------------

        time_col = next(

            (
                c
                for c in [
                    "timestamp",
                    "datetime",
                    "date_time",
                    "time_stamp",
                ]
                if c in x.columns
            ),

            None
        )


        if time_col is not None:

            idx = pd.to_datetime(
                x[time_col],
                utc=True,
                errors="coerce"
            )


        # --------------------------------
        # date + time
        # --------------------------------

        elif (
            "date" in x.columns
            and
            "time" in x.columns
        ):

            combined = (

                x["date"].astype(str)

                +

                " "

                +

                x["time"].astype(str)
            )


            idx = pd.to_datetime(
                combined,
                utc=True,
                errors="coerce"
            )


        # --------------------------------
        # date列だけ
        # --------------------------------

        elif "date" in x.columns:

            idx = pd.to_datetime(
                x["date"],
                utc=True,
                errors="coerce"
            )


        # --------------------------------
        # time列だけ
        # --------------------------------

        elif "time" in x.columns:

            idx = pd.to_datetime(
                x["time"],
                utc=True,
                errors="coerce"
            )


        # --------------------------------
        # 最初の列を試す
        # --------------------------------

        else:

            first_col = x.columns[0]

            test_idx = pd.to_datetime(
                x[first_col],
                utc=True,
                errors="coerce"
            )


            valid_ratio = (
                test_idx.notna().mean()
            )


            if valid_ratio < 0.80:

                raise ValueError(
                    "timestamp column not found"
                )


            idx = test_idx


    x.index = idx


    x = x.loc[
        ~x.index.isna()
    ].copy()


    # ========================================================
    # OHLC
    # ========================================================

    required = [
        "open",
        "high",
        "low",
        "close",
    ]


    missing = [
        c
        for c in required
        if c not in x.columns
    ]


    if missing:

        raise ValueError(
            f"OHLC missing: {missing}"
        )


    x = x[
        required
    ].copy()


    for c in required:

        x[c] = pd.to_numeric(
            x[c],
            errors="coerce"
        )


    x = x.dropna(
        subset=required
    )


    x = x.sort_index()


    return x


# ============================================================
# 4. CSV候補探索
# ============================================================

print("=" * 120)
print("SEARCHING USDJPY CSV FILES")
print("=" * 120)

cwd = Path.cwd()

print(
    "Current directory:",
    cwd
)


search_roots = []

# 現在directory
search_roots.append(
    cwd
)

# 一つ上
if cwd.parent != cwd:

    search_roots.append(
        cwd.parent
    )


# Notebook内にPath/string変数があれば
# USDJPY関連directory候補として追加

for name, obj in list(globals().items()):

    try:

        if isinstance(
            obj,
            (
                str,
                Path
            )
        ):

            text = str(obj).lower()

            if (
                "usd" in text
                and
                "jpy" in text
            ):

                p = Path(obj)

                if p.exists():

                    if p.is_file():

                        search_roots.append(
                            p.parent
                        )

                    elif p.is_dir():

                        search_roots.append(
                            p
                        )

    except Exception:

        pass


# duplicate roots除去

unique_roots = []

seen_roots = set()


for r in search_roots:

    try:

        rr = r.resolve()

    except Exception:

        continue


    if rr in seen_roots:

        continue


    seen_roots.add(rr)

    unique_roots.append(rr)


print(
    "Search roots:"
)

for r in unique_roots:

    print(
        " -",
        r
    )


# ============================================================
# 5. CSV一覧
# ============================================================

excluded_dirs = {

    ".git",
    ".ipynb_checkpoints",

    "csv_source_audit",
    "data_integrity_audit",
    "source_contamination_audit",
}


all_csv = []


for root in unique_roots:

    try:

        for path in root.rglob("*.csv"):

            parts_lower = {
                p.lower()
                for p in path.parts
            }


            if any(
                d.lower() in parts_lower
                for d in excluded_dirs
            ):

                continue


            name_lower = (
                path.name.lower()
            )


            # USDJPYっぽいものを優先
            if (

                "usdjpy" in name_lower

                or

                "usd_jpy" in name_lower

                or

                "usd-jpy" in name_lower

                or

                "jpy=x" in name_lower
            ):

                all_csv.append(
                    path
                )


            if len(all_csv) >= MAX_FILES:

                break


    except Exception:

        pass


# ------------------------------------------------------------
# USDJPY名が見つからない場合だけ全CSV
# ------------------------------------------------------------

if not all_csv:

    print()
    print(
        "[WARNING] USDJPYという名前のCSVが見つからないため、"
        "working directory内のCSVを探索します。"
    )


    for root in unique_roots:

        try:

            for path in root.rglob("*.csv"):

                parts_lower = {
                    p.lower()
                    for p in path.parts
                }


                if any(
                    d.lower() in parts_lower
                    for d in excluded_dirs
                ):

                    continue


                all_csv.append(
                    path
                )


                if len(all_csv) >= MAX_FILES:

                    break

        except Exception:

            pass


# duplicate path除去

csv_paths = []

seen = set()


for p in all_csv:

    try:

        rp = p.resolve()

    except Exception:

        continue


    if rp in seen:

        continue


    seen.add(rp)

    csv_paths.append(rp)


print()
print(
    "CSV candidates:",
    len(csv_paths)
)


if not csv_paths:

    raise RuntimeError(
        "CSVファイルが発見できませんでした。"
    )


# ============================================================
# 6. 1ファイル監査関数
# ============================================================

def audit_one_csv(path):

    file_result = {

        "file":
            str(path),

        "filename":
            path.name,

        "status":
            "UNKNOWN",

        "rows_raw":
            np.nan,

        "rows_valid":
            np.nan,

        "start":
            pd.NaT,

        "end":
            pd.NaT,

        "exact_grid_rows":
            0,

        "offgrid_rows":
            0,

        "duplicate_timestamps":
            0,

        "invalid_ohlc":
            0,

        "extreme_2pct_gaps":
            0,

        "extreme_5pct_gaps":
            0,

        "immediate_reversals":
            0,

        "reversal_ratio":
            np.nan,

        "max_abs_gap":
            np.nan,

        "median_intrabar_abs_return_extreme":
            np.nan,

        "min_close":
            np.nan,

        "max_close":
            np.nan,

        "daily_span_gt5pct_days":
            0,

        "suspicious_score":
            0,
    }


    event_rows = []

    day_rows = []


    try:

        raw = read_csv_flexible(
            path
        )


        file_result[
            "rows_raw"
        ] = len(raw)


        x = normalize_csv_ohlc(
            raw,
            path
        )


        file_result[
            "rows_valid"
        ] = len(x)


        if len(x) < 2:

            file_result[
                "status"
            ] = "TOO_FEW_ROWS"

            return (
                file_result,
                pd.DataFrame(),
                pd.DataFrame(),
                None
            )


        file_result[
            "start"
        ] = x.index.min()


        file_result[
            "end"
        ] = x.index.max()


        file_result[
            "min_close"
        ] = x["close"].min()


        file_result[
            "max_close"
        ] = x["close"].max()


        # ====================================================
        # grid
        # ====================================================

        exact_mask = (

            (x.index.minute % 15 == 0)

            &

            (x.index.second == 0)

            &

            (x.index.microsecond == 0)
        )


        file_result[
            "exact_grid_rows"
        ] = int(
            exact_mask.sum()
        )


        file_result[
            "offgrid_rows"
        ] = int(
            (~exact_mask).sum()
        )


        exact = x.loc[
            exact_mask
        ].copy()


        # ====================================================
        # duplicates
        # ====================================================

        duplicate_count = int(
            exact.index.duplicated().sum()
        )


        file_result[
            "duplicate_timestamps"
        ] = duplicate_count


        exact = exact.loc[
            ~exact.index.duplicated(
                keep="first"
            )
        ].copy()


        # ====================================================
        # OHLC validity
        # ====================================================

        invalid_high = (

            exact["high"]

            <

            exact[
                [
                    "open",
                    "close",
                    "low"
                ]
            ].max(
                axis=1
            )
        )


        invalid_low = (

            exact["low"]

            >

            exact[
                [
                    "open",
                    "close",
                    "high"
                ]
            ].min(
                axis=1
            )
        )


        nonpositive = (

            exact[
                [
                    "open",
                    "high",
                    "low",
                    "close"
                ]
            ]

            <= 0

        ).any(axis=1)


        invalid = (

            invalid_high

            |

            invalid_low

            |

            nonpositive
        )


        file_result[
            "invalid_ohlc"
        ] = int(
            invalid.sum()
        )


        if len(exact) < 2:

            file_result[
                "status"
            ] = "NO_EXACT_15M_DATA"

            return (
                file_result,
                pd.DataFrame(),
                pd.DataFrame(),
                exact
            )


        # ====================================================
        # sequential gap
        # ====================================================

        time_series = pd.Series(
            exact.index,
            index=exact.index
        )


        prev_time = (
            time_series.shift(1)
        )


        next_time = (
            time_series.shift(-1)
        )


        prev_close = (
            exact["close"].shift(1)
        )


        contiguous_prev = (

            (
                time_series
                -
                prev_time
            )

            ==

            pd.Timedelta(
                minutes=15
            )
        )


        contiguous_next = (

            (
                next_time
                -
                time_series
            )

            ==

            pd.Timedelta(
                minutes=15
            )
        )


        exact[
            "open_gap"
        ] = np.where(

            contiguous_prev,

            exact["open"]
            /
            prev_close
            -
            1,

            np.nan
        )


        exact[
            "close_gap"
        ] = np.where(

            contiguous_prev,

            exact["close"]
            /
            prev_close
            -
            1,

            np.nan
        )


        exact[
            "bar_return"
        ] = (

            exact["close"]
            /
            exact["open"]
            -
            1
        )


        exact[
            "intrabar_range"
        ] = (

            exact["high"]
            /
            exact["low"]
            -
            1
        )


        exact[
            "extreme_2"
        ] = (

            exact[
                "open_gap"
            ].abs()

            >

            EXTREME_2_THRESHOLD
        )


        exact[
            "extreme_5"
        ] = (

            exact[
                "open_gap"
            ].abs()

            >

            GAP_THRESHOLD
        )


        exact[
            "next_gap"
        ] = (

            exact[
                "open_gap"
            ]
            .shift(-1)
        )


        exact[
            "immediate_reversal"
        ] = (

            exact[
                "extreme_5"
            ]

            &

            contiguous_next

            &

            (
                exact[
                    "next_gap"
                ].abs()

                >

                GAP_THRESHOLD
            )

            &

            (
                np.sign(
                    exact[
                        "open_gap"
                    ]
                )

                !=

                np.sign(
                    exact[
                        "next_gap"
                    ]
                )
            )
        )


        # ====================================================
        # stats
        # ====================================================

        extreme5 = exact.loc[
            exact[
                "extreme_5"
            ]
        ].copy()


        reversals = exact.loc[
            exact[
                "immediate_reversal"
            ]
        ].copy()


        n_extreme = len(
            extreme5
        )


        n_reverse = len(
            reversals
        )


        file_result[
            "extreme_2pct_gaps"
        ] = int(

            exact[
                "extreme_2"
            ].sum()
        )


        file_result[
            "extreme_5pct_gaps"
        ] = int(
            n_extreme
        )


        file_result[
            "immediate_reversals"
        ] = int(
            n_reverse
        )


        file_result[
            "reversal_ratio"
        ] = (

            n_reverse
            /
            n_extreme

            if n_extreme

            else 0.0
        )


        file_result[
            "max_abs_gap"
        ] = (

            float(

                exact[
                    "open_gap"
                ].abs().max()
            )

            if exact[
                "open_gap"
            ].notna().any()

            else np.nan
        )


        file_result[
            "median_intrabar_abs_return_extreme"
        ] = (

            float(

                extreme5[
                    "bar_return"
                ].abs().median()
            )

            if n_extreme

            else np.nan
        )


        # ====================================================
        # daily contamination
        # ====================================================

        d = exact.copy()

        d[
            "date"
        ] = d.index.normalize()


        daily = (

            d

            .groupby(
                "date"
            )

            .agg(

                rows=(
                    "close",
                    "size"
                ),

                min_close=(
                    "close",
                    "min"
                ),

                max_close=(
                    "close",
                    "max"
                ),

                extreme_5pct_gaps=(
                    "extreme_5",
                    "sum"
                ),

                immediate_reversals=(
                    "immediate_reversal",
                    "sum"
                ),
            )

            .reset_index()
        )


        daily[
            "price_span"
        ] = (

            daily[
                "max_close"
            ]

            /

            daily[
                "min_close"
            ]

            -

            1
        )


        suspicious_daily = daily.loc[

            (
                daily[
                    "extreme_5pct_gaps"
                ]
                >
                0
            )

            |

            (
                daily[
                    "price_span"
                ]
                >
                0.05
            )

        ].copy()


        file_result[
            "daily_span_gt5pct_days"
        ] = int(
            len(
                suspicious_daily
            )
        )


        if len(
            suspicious_daily
        ):

            suspicious_daily[
                "file"
            ] = str(
                path
            )


            suspicious_daily[
                "filename"
            ] = path.name


            day_rows.append(
                suspicious_daily
            )


        # ====================================================
        # event rows
        # ====================================================

        if n_extreme:

            events = exact.loc[
                exact[
                    "extreme_5"
                ]
            ].copy()


            events[
                "file"
            ] = str(
                path
            )


            events[
                "filename"
            ] = path.name


            events[
                "prev_close"
            ] = prev_close.reindex(
                events.index
            )


            event_rows.append(
                events[
                    [
                        "file",
                        "filename",

                        "open",
                        "high",
                        "low",
                        "close",

                        "prev_close",

                        "open_gap",
                        "close_gap",

                        "bar_return",
                        "intrabar_range",

                        "next_gap",

                        "immediate_reversal",
                    ]
                ]
            )


        # ====================================================
        # suspicious score
        # ====================================================

        score = 0


        score += (
            file_result[
                "extreme_5pct_gaps"
            ]
            *
            5
        )


        score += (
            file_result[
                "immediate_reversals"
            ]
            *
            10
        )


        score += (
            file_result[
                "daily_span_gt5pct_days"
            ]
            *
            3
        )


        score += (
            file_result[
                "invalid_ohlc"
            ]
            *
            10
        )


        score += (
            file_result[
                "duplicate_timestamps"
            ]
            *
            2
        )


        file_result[
            "suspicious_score"
        ] = score


        if (
            n_extreme > 0
            and
            file_result[
                "reversal_ratio"
            ] >= 0.20
            and
            np.isfinite(
                file_result[
                    "median_intrabar_abs_return_extreme"
                ]
            )
            and
            file_result[
                "median_intrabar_abs_return_extreme"
            ] < 0.01
        ):

            file_result[
                "status"
            ] = "STRONG_MIXED_SERIES"

        elif n_extreme > 0:

            file_result[
                "status"
            ] = "SUSPICIOUS"

        elif (
            file_result[
                "invalid_ohlc"
            ] > 0
        ):

            file_result[
                "status"
            ] = "INVALID_OHLC"

        else:

            file_result[
                "status"
            ] = "OK"


        return (

            file_result,

            (
                pd.concat(
                    event_rows
                )
                if event_rows
                else pd.DataFrame()
            ),

            (
                pd.concat(
                    day_rows
                )
                if day_rows
                else pd.DataFrame()
            ),

            exact,
        )


    except Exception as e:

        file_result[
            "status"
        ] = "PARSE_ERROR"


        file_result[
            "error"
        ] = str(
            e
        )


        return (
            file_result,
            pd.DataFrame(),
            pd.DataFrame(),
            None
        )


# ============================================================
# 7. 全CSV監査
# ============================================================

summary_rows = []

event_frames = []

day_frames = []

exact_files = []


print()
print("=" * 120)
print("AUDITING FILES")
print("=" * 120)


for i, path in enumerate(
    csv_paths,
    start=1
):

    print(
        f"[{i}/{len(csv_paths)}] {path.name}",
        end=" ... "
    )


    (
        result,
        events,
        days,
        exact
    ) = audit_one_csv(
        path
    )


    summary_rows.append(
        result
    )


    if len(events):

        event_frames.append(
            events
        )


    if len(days):

        day_frames.append(
            days
        )


    if exact is not None and len(exact):

        exact_files.append(
            {
                "file":
                    str(path),

                "filename":
                    path.name,

                "start":
                    exact.index.min(),

                "end":
                    exact.index.max(),

                "first_open":
                    float(
                        exact.iloc[0][
                            "open"
                        ]
                    ),

                "first_close":
                    float(
                        exact.iloc[0][
                            "close"
                        ]
                    ),

                "last_open":
                    float(
                        exact.iloc[-1][
                            "open"
                        ]
                    ),

                "last_close":
                    float(
                        exact.iloc[-1][
                            "close"
                        ]
                    ),
            }
        )


    print(
        result[
            "status"
        ]
    )


FILE_AUDIT = pd.DataFrame(
    summary_rows
)


EXTREME_EVENTS = (

    pd.concat(
        event_frames
    )

    if event_frames

    else pd.DataFrame()
)


SUSPICIOUS_DAYS = (

    pd.concat(
        day_frames,
        ignore_index=True
    )

    if day_frames

    else pd.DataFrame()
)


# ============================================================
# 8. CSV間 boundary check
# ============================================================

BOUNDARY_ROWS = []


if exact_files:

    boundaries = (

        pd.DataFrame(
            exact_files
        )

        .sort_values(
            "start"
        )

        .reset_index(
            drop=True
        )
    )


    for i in range(
        1,
        len(boundaries)
    ):

        prev = boundaries.iloc[
            i - 1
        ]


        curr = boundaries.iloc[
            i
        ]


        time_gap = (

            curr[
                "start"
            ]

            -

            prev[
                "end"
            ]
        )


        boundary_return = (

            curr[
                "first_open"
            ]

            /

            prev[
                "last_close"
            ]

            -

            1
        )


        BOUNDARY_ROWS.append(
            {
                "previous_file":
                    prev[
                        "filename"
                    ],

                "next_file":
                    curr[
                        "filename"
                    ],

                "previous_end":
                    prev[
                        "end"
                    ],

                "next_start":
                    curr[
                        "start"
                    ],

                "time_gap":
                    time_gap,

                "previous_last_close":
                    prev[
                        "last_close"
                    ],

                "next_first_open":
                    curr[
                        "first_open"
                    ],

                "boundary_return":
                    boundary_return,

                "abs_boundary_return":
                    abs(
                        boundary_return
                    ),

                "boundary_gt5pct":
                    abs(
                        boundary_return
                    )
                    >
                    GAP_THRESHOLD,
            }
        )


BOUNDARY_AUDIT = pd.DataFrame(
    BOUNDARY_ROWS
)


# ============================================================
# 9. Suspicious files
# ============================================================

if len(FILE_AUDIT):

    SUSPICIOUS_FILES = FILE_AUDIT.loc[

        FILE_AUDIT[
            "status"
        ].isin(
            [
                "STRONG_MIXED_SERIES",
                "SUSPICIOUS",
                "INVALID_OHLC",
            ]
        )

    ].copy()


    SUSPICIOUS_FILES = SUSPICIOUS_FILES.sort_values(

        [
            "suspicious_score",
            "extreme_5pct_gaps",
            "immediate_reversals",
        ],

        ascending=False
    )


else:

    SUSPICIOUS_FILES = pd.DataFrame()


# ============================================================
# 10. Monthly source attribution
# ============================================================

MONTH_SOURCE_ROWS = []


if len(EXTREME_EVENTS):

    e = EXTREME_EVENTS.copy()


    e[
        "month"
    ] = e.index.strftime(
        "%Y-%m"
    )


    temp = (

        e

        .groupby(
            [
                "filename",
                "month"
            ]
        )

        .agg(

            extreme_gaps=(
                "open_gap",
                "size"
            ),

            immediate_reversals=(
                "immediate_reversal",
                "sum"
            ),

            max_abs_gap=(
                "open_gap",
                lambda s:
                s.abs().max()
            ),

            median_intrabar_abs_return=(
                "bar_return",
                lambda s:
                s.abs().median()
            ),
        )

        .reset_index()
    )


    MONTH_SOURCE_AUDIT = temp.sort_values(

        [
            "extreme_gaps",
            "immediate_reversals"
        ],

        ascending=False
    )


else:

    MONTH_SOURCE_AUDIT = (
        pd.DataFrame()
    )


# ============================================================
# 11. 表示
# ============================================================

show_table(
    "CSV SOURCE AUDIT SUMMARY",
    FILE_AUDIT.sort_values(
        "suspicious_score",
        ascending=False
    ),
    200
)


show_table(
    "FLAGGED / SUSPICIOUS FILES",
    SUSPICIOUS_FILES,
    100
)


show_table(
    "EXTREME >5% EVENTS WITH SOURCE FILE",
    (
        EXTREME_EVENTS.assign(
            abs_gap=
            EXTREME_EVENTS[
                "open_gap"
            ].abs()
        )
        .sort_values(
            "abs_gap",
            ascending=False
        )
        .drop(
            columns="abs_gap"
        )
        if len(
            EXTREME_EVENTS
        )
        else pd.DataFrame()
    ),
    150
)


show_table(
    "SUSPICIOUS DAYS BY SOURCE FILE",
    SUSPICIOUS_DAYS,
    150
)


show_table(
    "SUSPICIOUS MONTHS BY SOURCE FILE",
    MONTH_SOURCE_AUDIT,
    150
)


show_table(
    "CSV FILE BOUNDARY AUDIT",
    BOUNDARY_AUDIT.sort_values(
        "abs_boundary_return",
        ascending=False
    )
    if len(
        BOUNDARY_AUDIT
    )
    else BOUNDARY_AUDIT,
    100
)


# ============================================================
# 12. Automatic diagnosis
# ============================================================

strong_files = (

    FILE_AUDIT.loc[

        FILE_AUDIT[
            "status"
        ]
        ==
        "STRONG_MIXED_SERIES"

    ]

    if len(
        FILE_AUDIT
    )

    else pd.DataFrame()
)


suspicious_files = (

    FILE_AUDIT.loc[

        FILE_AUDIT[
            "status"
        ]
        ==
        "SUSPICIOUS"

    ]

    if len(
        FILE_AUDIT
    )

    else pd.DataFrame()
)


boundary_bad = (

    BOUNDARY_AUDIT.loc[

        BOUNDARY_AUDIT[
            "boundary_gt5pct"
        ]
        ==
        True

    ]

    if (
        len(
            BOUNDARY_AUDIT
        )
        and
        "boundary_gt5pct"
        in
        BOUNDARY_AUDIT.columns
    )

    else pd.DataFrame()
)


print()
print("=" * 120)
print("FINAL SOURCE FILE DIAGNOSIS")
print("=" * 120)

print(
    "CSV files checked:",
    len(
        FILE_AUDIT
    )
)

print(
    "Strong mixed-series files:",
    len(
        strong_files
    )
)

print(
    "Other suspicious files:",
    len(
        suspicious_files
    )
)

print(
    "Suspicious >5% file boundaries:",
    len(
        boundary_bad
    )
)


if len(
    strong_files
):

    FINAL_SOURCE_DECISION = (
        "CONTAMINATED_SOURCE_FILES_IDENTIFIED"
    )


elif len(
    suspicious_files
):

    FINAL_SOURCE_DECISION = (
        "SUSPICIOUS_SOURCE_FILES_IDENTIFIED"
    )


elif len(
    boundary_bad
):

    FINAL_SOURCE_DECISION = (
        "CSV_BOUNDARY_CONTAMINATION_IDENTIFIED"
    )


else:

    FINAL_SOURCE_DECISION = (
        "NO_CONTAMINATED_FILE_IDENTIFIED"
    )


print()
print(
    "FINAL DECISION:",
    FINAL_SOURCE_DECISION
)


if len(
    strong_files
):

    print()
    print(
        "最も疑わしいCSV:"
    )


    for row in strong_files.head(
        20
    ).itertuples():

        print(
            " -",
            row.filename,
            "| >5% gaps:",
            row.extreme_5pct_gaps,
            "| reversals:",
            row.immediate_reversals,
            "| ratio:",
            f"{row.reversal_ratio:.2%}",
            "| period:",
            row.start,
            "->",
            row.end,
        )


# ============================================================
# 13. 保存
# ============================================================

FILE_AUDIT.to_csv(
    OUTPUT_DIR /
    "file_audit_summary.csv",
    index=False
)


SUSPICIOUS_FILES.to_csv(
    OUTPUT_DIR /
    "suspicious_files.csv",
    index=False
)


EXTREME_EVENTS.to_csv(
    OUTPUT_DIR /
    "extreme_events_with_source.csv",
    index_label="timestamp"
)


SUSPICIOUS_DAYS.to_csv(
    OUTPUT_DIR /
    "suspicious_days_by_source.csv",
    index=False
)


MONTH_SOURCE_AUDIT.to_csv(
    OUTPUT_DIR /
    "suspicious_months_by_source.csv",
    index=False
)


BOUNDARY_AUDIT.to_csv(
    OUTPUT_DIR /
    "csv_boundary_audit.csv",
    index=False
)


# Notebookに残す

CSV_SOURCE_AUDIT = (
    FILE_AUDIT.copy()
)

CSV_SOURCE_SUSPICIOUS = (
    SUSPICIOUS_FILES.copy()
)

CSV_SOURCE_EXTREME_EVENTS = (
    EXTREME_EVENTS.copy()
)

CSV_SOURCE_MONTHS = (
    MONTH_SOURCE_AUDIT.copy()
)

CSV_SOURCE_BOUNDARIES = (
    BOUNDARY_AUDIT.copy()
)

CSV_SOURCE_DECISION = (
    FINAL_SOURCE_DECISION
)


print()
print("=" * 120)
print("CSV SOURCE AUDIT COMPLETE")
print("=" * 120)

print(
    "Saved folder:",
    OUTPUT_DIR
)

print()
print(
    "Notebook variables:"
)

print(
    "CSV_SOURCE_AUDIT"
)

print(
    "CSV_SOURCE_SUSPICIOUS"
)

print(
    "CSV_SOURCE_EXTREME_EVENTS"
)

print(
    "CSV_SOURCE_MONTHS"
)

print(
    "CSV_SOURCE_BOUNDARIES"
)

print(
    "CSV_SOURCE_DECISION"
)


## 元セルindex 64
構文状態：valid


In [ ]:
# ============================================================
# FINAL CLEAN DATASET REBUILD
# USDJPY yearly 15m files ONLY
#
# 使用:
#   usdjpy_15m_2016.csv
#   usdjpy_15m_2017.csv
#   ...
#   usdjpy_15m_2026.csv
#
# 使用禁止:
#   usdjpy_15m_2016_2026.csv
#   usdjpy_5m.csv
#
# bars はまだ上書きしない
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 200)

YEARS = list(range(2016, 2027))

OUTPUT_DIR = Path("clean_dataset_rebuild")
OUTPUT_DIR.mkdir(exist_ok=True)


# ============================================================
# 1. CSV読込
# ============================================================

def read_csv_safe(path):

    attempts = []

    for kwargs in [

        dict(
            low_memory=False
        ),

        dict(
            sep=None,
            engine="python"
        ),

        dict(
            sep=";",
            low_memory=False
        ),

        dict(
            sep="\t",
            low_memory=False
        ),
    ]:

        try:

            df = pd.read_csv(
                path,
                **kwargs
            )

            attempts.append(
                df
            )

        except Exception:

            pass


    if not attempts:

        raise RuntimeError(
            f"CSV read failed: {path}"
        )


    attempts.sort(
        key=lambda d: d.shape[1],
        reverse=True
    )

    return attempts[0]


# ============================================================
# 2. OHLC正規化
# ============================================================

def normalize_ohlc(df, path):

    x = df.copy()

    x.columns = [
        str(c).strip().lower()
        for c in x.columns
    ]


    aliases = {

        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",

        "bidopen": "open",
        "bidhigh": "high",
        "bidlow": "low",
        "bidclose": "close",

        "open_bid": "open",
        "high_bid": "high",
        "low_bid": "low",
        "close_bid": "close",

        "<open>": "open",
        "<high>": "high",
        "<low>": "low",
        "<close>": "close",

        "<date>": "date",
        "<time>": "time",
    }


    x = x.rename(
        columns={
            c: aliases.get(c, c)
            for c in x.columns
        }
    )


    # --------------------------------------------------------
    # Timestamp
    # --------------------------------------------------------

    if isinstance(
        x.index,
        pd.DatetimeIndex
    ):

        idx = pd.to_datetime(
            x.index,
            utc=True,
            errors="coerce"
        )

    else:

        timestamp_col = next(
            (
                c
                for c in [
                    "timestamp",
                    "datetime",
                    "date_time",
                ]
                if c in x.columns
            ),
            None
        )


        if timestamp_col is not None:

            idx = pd.to_datetime(
                x[timestamp_col],
                utc=True,
                errors="coerce"
            )


        elif (
            "date" in x.columns
            and
            "time" in x.columns
        ):

            idx = pd.to_datetime(

                x["date"].astype(str)
                +
                " "
                +
                x["time"].astype(str),

                utc=True,
                errors="coerce"
            )


        elif "date" in x.columns:

            idx = pd.to_datetime(
                x["date"],
                utc=True,
                errors="coerce"
            )


        else:

            # 最初の列をtimestamp候補として試す

            first = x.columns[0]

            idx_test = pd.to_datetime(
                x[first],
                utc=True,
                errors="coerce"
            )


            if idx_test.notna().mean() < 0.90:

                raise RuntimeError(
                    f"Timestamp column not found: {path}"
                )


            idx = idx_test


    x.index = idx

    x = x.loc[
        ~x.index.isna()
    ].copy()


    # --------------------------------------------------------
    # OHLC
    # --------------------------------------------------------

    required = [
        "open",
        "high",
        "low",
        "close",
    ]


    missing = [
        c
        for c in required
        if c not in x.columns
    ]


    if missing:

        raise RuntimeError(
            f"{path.name}: missing {missing}"
        )


    x = x[
        required
    ].copy()


    for c in required:

        x[c] = pd.to_numeric(
            x[c],
            errors="coerce"
        )


    x = x.dropna()

    x = x.sort_index()

    return x


# ============================================================
# 3. 正確な年別15分CSVだけを探す
# ============================================================

print("=" * 100)
print("SEARCH YEARLY USDJPY 15M FILES")
print("=" * 100)


# 以前のaudit結果を利用できれば優先
candidate_paths = []


if "CSV_SOURCE_AUDIT" in globals():

    audit = CSV_SOURCE_AUDIT.copy()


    if "file" in audit.columns:

        for value in audit[
            "file"
        ].dropna():

            p = Path(
                str(value)
            )

            if p.exists():

                candidate_paths.append(
                    p
                )


# fallback
if not candidate_paths:

    for p in Path.cwd().rglob(
        "*.csv"
    ):

        candidate_paths.append(
            p
        )


# duplicate除去
unique_paths = []

seen = set()


for p in candidate_paths:

    try:

        rp = p.resolve()

    except Exception:

        continue


    if rp in seen:

        continue


    seen.add(
        rp
    )

    unique_paths.append(
        rp
    )


# ------------------------------------------------------------
# EXACT filenameのみ
#
# usdjpy_15m_2025.csv
# の形だけ
# ------------------------------------------------------------

pattern = re.compile(
    r"^usdjpy_15m_(20\d{2})\.csv$",
    re.IGNORECASE
)


year_files = {}


for path in unique_paths:

    match = pattern.match(
        path.name
    )


    if not match:

        continue


    year = int(
        match.group(1)
    )


    if year not in YEARS:

        continue


    year_files.setdefault(
        year,
        []
    ).append(
        path
    )


print()

for year in YEARS:

    files = year_files.get(
        year,
        []
    )

    print(
        year,
        "->",
        len(files),
        "file(s)"
    )

    for p in files:

        print(
            "   ",
            p
        )


# ============================================================
# 4. 1年1ファイルであることを要求
# ============================================================

missing_years = [

    year
    for year in YEARS

    if len(
        year_files.get(
            year,
            []
        )
    ) == 0
]


duplicate_years = [

    year
    for year in YEARS

    if len(
        year_files.get(
            year,
            []
        )
    ) > 1
]


if missing_years:

    raise RuntimeError(
        f"年別15m CSVが不足しています: {missing_years}"
    )


if duplicate_years:

    print()
    print(
        "WARNING:"
    )

    print(
        "同じ年のCSVが複数あります:",
        duplicate_years
    )

    print(
        "安全のため自動選択しません。"
    )

    raise RuntimeError(
        "Duplicate yearly files detected."
    )


# ============================================================
# 5. 各年を独立読込
# ============================================================

frames = []

YEAR_AUDIT_ROWS = []


for year in YEARS:

    path = year_files[
        year
    ][0]


    raw = read_csv_safe(
        path
    )


    x = normalize_ohlc(
        raw,
        path
    )


    # --------------------------------------------------------
    # そのCSVが本当にその年だけか
    # --------------------------------------------------------

    wrong_year = (
        x.index.year
        != year
    )


    wrong_year_count = int(
        wrong_year.sum()
    )


    # --------------------------------------------------------
    # exact 15m grid
    # --------------------------------------------------------

    exact_grid = (

        (x.index.minute % 15 == 0)

        &

        (x.index.second == 0)

        &

        (x.index.microsecond == 0)
    )


    offgrid_count = int(
        (~exact_grid).sum()
    )


    duplicate_count = int(
        x.index.duplicated().sum()
    )


    # --------------------------------------------------------
    # OHLC validity
    # --------------------------------------------------------

    invalid_high = (

        x["high"]

        <

        x[
            [
                "open",
                "close",
                "low"
            ]
        ].max(axis=1)
    )


    invalid_low = (

        x["low"]

        >

        x[
            [
                "open",
                "close",
                "high"
            ]
        ].min(axis=1)
    )


    invalid_nonpositive = (

        x[
            [
                "open",
                "high",
                "low",
                "close"
            ]
        ]

        <= 0

    ).any(axis=1)


    invalid_count = int(

        (
            invalid_high
            |
            invalid_low
            |
            invalid_nonpositive
        ).sum()
    )


    # --------------------------------------------------------
    # HARD FAIL
    # --------------------------------------------------------

    if wrong_year_count > 0:

        raise RuntimeError(
            f"{path.name}: "
            f"{wrong_year_count} rows belong to another year."
        )


    if offgrid_count > 0:

        raise RuntimeError(
            f"{path.name}: "
            f"{offgrid_count} off-grid rows found."
        )


    if duplicate_count > 0:

        raise RuntimeError(
            f"{path.name}: "
            f"{duplicate_count} duplicate timestamps."
        )


    if invalid_count > 0:

        raise RuntimeError(
            f"{path.name}: "
            f"{invalid_count} invalid OHLC rows."
        )


    # --------------------------------------------------------
    # source info
    # --------------------------------------------------------

    x["source_year"] = year

    frames.append(
        x
    )


    YEAR_AUDIT_ROWS.append(
        {
            "year":
                year,

            "file":
                str(path),

            "rows":
                len(x),

            "start":
                x.index.min(),

            "end":
                x.index.max(),

            "first_open":
                x.iloc[0]["open"],

            "last_close":
                x.iloc[-1]["close"],

            "offgrid":
                offgrid_count,

            "duplicates":
                duplicate_count,

            "invalid_ohlc":
                invalid_count,
        }
    )


YEAR_AUDIT = pd.DataFrame(
    YEAR_AUDIT_ROWS
)


# ============================================================
# 6. 年別ファイルだけを結合
# ============================================================

REBUILT = (

    pd.concat(
        frames,
        axis=0
    )

    .sort_index()
)


# source_yearは検査用に残す
# モデルへ入れるbarsでは後で落とす


# ============================================================
# 7. 全体duplicate / ordering
# ============================================================

global_duplicate_count = int(
    REBUILT.index.duplicated().sum()
)


if global_duplicate_count > 0:

    duplicate_rows = REBUILT.loc[
        REBUILT.index.duplicated(
            keep=False
        )
    ]

    print(
        duplicate_rows.head(30)
    )

    raise RuntimeError(
        f"Yearly files overlap: "
        f"{global_duplicate_count} duplicate timestamps."
    )


if not REBUILT.index.is_monotonic_increasing:

    raise RuntimeError(
        "Timestamp ordering failure."
    )


# ============================================================
# 8. 年境界
# ============================================================

BOUNDARY_ROWS = []


for i in range(
    1,
    len(YEAR_AUDIT)
):

    prev = YEAR_AUDIT.iloc[
        i - 1
    ]

    curr = YEAR_AUDIT.iloc[
        i
    ]


    gap = (
        curr["start"]
        -
        prev["end"]
    )


    price_return = (

        curr["first_open"]
        /
        prev["last_close"]
        -
        1
    )


    BOUNDARY_ROWS.append(
        {
            "from_year":
                int(
                    prev["year"]
                ),

            "to_year":
                int(
                    curr["year"]
                ),

            "previous_end":
                prev["end"],

            "next_start":
                curr["start"],

            "time_gap":
                gap,

            "previous_close":
                prev["last_close"],

            "next_open":
                curr["first_open"],

            "boundary_return":
                price_return,

            "abs_boundary_return":
                abs(
                    price_return
                ),
        }
    )


BOUNDARY_AUDIT = pd.DataFrame(
    BOUNDARY_ROWS
)


# ============================================================
# 9. 連続15分bar間のjump
# ============================================================

time_series = pd.Series(
    REBUILT.index,
    index=REBUILT.index
)


prev_time = (
    time_series.shift(1)
)


prev_close = (
    REBUILT[
        "close"
    ].shift(1)
)


contiguous = (

    (
        time_series
        -
        prev_time
    )

    ==

    pd.Timedelta(
        minutes=15
    )
)


REBUILT[
    "audit_open_gap"
] = np.where(

    contiguous,

    REBUILT[
        "open"
    ]

    /

    prev_close

    -

    1,

    np.nan
)


REBUILT[
    "audit_bar_return"
] = (

    REBUILT[
        "close"
    ]

    /

    REBUILT[
        "open"
    ]

    -

    1
)


REBUILT[
    "audit_intrabar_range"
] = (

    REBUILT[
        "high"
    ]

    /

    REBUILT[
        "low"
    ]

    -

    1
)


EXTREME_5 = REBUILT.loc[

    REBUILT[
        "audit_open_gap"
    ].abs()

    >

    0.05

].copy()


EXTREME_2 = REBUILT.loc[

    REBUILT[
        "audit_open_gap"
    ].abs()

    >

    0.02

].copy()


# ============================================================
# 10. Immediate reversal
# ============================================================

REBUILT[
    "audit_next_gap"
] = (

    REBUILT[
        "audit_open_gap"
    ]
    .shift(-1)
)


next_time = (
    time_series.shift(-1)
)


contiguous_next = (

    (
        next_time
        -
        time_series
    )

    ==

    pd.Timedelta(
        minutes=15
    )
)


REBUILT[
    "audit_immediate_reversal"
] = (

    (
        REBUILT[
            "audit_open_gap"
        ].abs()

        >

        0.05
    )

    &

    (
        REBUILT[
            "audit_next_gap"
        ].abs()

        >

        0.05
    )

    &

    contiguous_next

    &

    (
        np.sign(
            REBUILT[
                "audit_open_gap"
            ]
        )

        !=

        np.sign(
            REBUILT[
                "audit_next_gap"
            ]
        )
    )
)


REVERSALS = REBUILT.loc[
    REBUILT[
        "audit_immediate_reversal"
    ]
].copy()


# ============================================================
# 11. Daily spanはWarningだけ
#
# 1日5%以上は実際に起こる可能性があるため
# FAIL条件にはしない
# ============================================================

daily_temp = REBUILT.copy()

daily_temp[
    "date"
] = (
    daily_temp.index.normalize()
)


DAILY = (

    daily_temp

    .groupby(
        "date"
    )

    .agg(

        rows=(
            "close",
            "size"
        ),

        min_close=(
            "close",
            "min"
        ),

        max_close=(
            "close",
            "max"
        ),
    )

    .reset_index()
)


DAILY[
    "price_span"
] = (

    DAILY[
        "max_close"
    ]

    /

    DAILY[
        "min_close"
    ]

    -

    1
)


LARGE_DAILY_SPAN = DAILY.loc[

    DAILY[
        "price_span"
    ]
    >
    0.05

].copy()


# ============================================================
# 12. 旧aggregateと比較
# ============================================================

AGGREGATE_COMPARISON = pd.DataFrame()


aggregate_candidates = [

    p
    for p in unique_paths

    if p.name.lower()
    ==
    "usdjpy_15m_2016_2026.csv"
]


if len(
    aggregate_candidates
) == 1:

    agg_path = aggregate_candidates[
        0
    ]


    agg_raw = read_csv_safe(
        agg_path
    )


    agg = normalize_ohlc(
        agg_raw,
        agg_path
    )


    common = (

        REBUILT.index

        .intersection(
            agg.index
        )
    )


    comp = pd.DataFrame(
        index=common
    )


    for c in [
        "open",
        "high",
        "low",
        "close",
    ]:

        comp[
            f"yearly_{c}"
        ] = REBUILT.loc[
            common,
            c
        ]


        comp[
            f"aggregate_{c}"
        ] = agg.loc[
            common,
            c
        ]


        comp[
            f"{c}_diff"
        ] = (

            comp[
                f"yearly_{c}"
            ]

            -

            comp[
                f"aggregate_{c}"
            ]

        ).abs()


    comp[
        "max_diff"
    ] = comp[
        [
            "open_diff",
            "high_diff",
            "low_diff",
            "close_diff",
        ]
    ].max(axis=1)


    AGGREGATE_COMPARISON = (

        comp.loc[
            comp[
                "max_diff"
            ]
            >
            1e-10
        ]

        .sort_values(
            "max_diff",
            ascending=False
        )
    )


# ============================================================
# 13. Summary
# ============================================================

SUMMARY = pd.DataFrame(
    [
        {
            "metric":
                "Years loaded",

            "value":
                len(
                    YEAR_AUDIT
                ),
        },

        {
            "metric":
                "Total rows",

            "value":
                len(
                    REBUILT
                ),
        },

        {
            "metric":
                "Duplicate timestamps",

            "value":
                global_duplicate_count,
        },

        {
            "metric":
                "Contiguous gaps >2%",

            "value":
                len(
                    EXTREME_2
                ),
        },

        {
            "metric":
                "Contiguous gaps >5%",

            "value":
                len(
                    EXTREME_5
                ),
        },

        {
            "metric":
                "Immediate opposite >5% reversals",

            "value":
                len(
                    REVERSALS
                ),
        },

        {
            "metric":
                "Days with >5% total span",

            "value":
                len(
                    LARGE_DAILY_SPAN
                ),
        },

        {
            "metric":
                "Aggregate-vs-yearly differing timestamps",

            "value":
                len(
                    AGGREGATE_COMPARISON
                ),
        },
    ]
)


# ============================================================
# 14. Decision
# ============================================================

hard_failures = []


if len(
    YEAR_AUDIT
) != len(
    YEARS
):

    hard_failures.append(
        "Not all years loaded."
    )


if global_duplicate_count > 0:

    hard_failures.append(
        "Duplicate timestamps."
    )


if len(
    EXTREME_5
) > 0:

    hard_failures.append(
        "Contiguous >5% jumps remain."
    )


if len(
    REVERSALS
) > 0:

    hard_failures.append(
        "Immediate opposite >5% reversals remain."
    )


# 年境界がマイナスなら異常
if len(
    BOUNDARY_AUDIT
):

    bad_negative_boundaries = (

        BOUNDARY_AUDIT[
            "time_gap"
        ]

        <=

        pd.Timedelta(0)
    )


    if bad_negative_boundaries.any():

        hard_failures.append(
            "Overlapping or reversed yearly boundaries."
        )


if hard_failures:

    FINAL_DECISION = (
        "REBUILD_REQUIRES_REVIEW"
    )

else:

    FINAL_DECISION = (
        "CLEAN_YEARLY_DATASET_READY"
    )


# ============================================================
# 15. 表示
# ============================================================

def show(
    title,
    df,
    n=100
):

    print()
    print("=" * 105)
    print(title)
    print("=" * 105)

    if df is None or len(df) == 0:

        print(
            "No data"
        )

        return


    try:

        display(
            df.head(n)
        )

    except Exception:

        print(
            df.head(n).to_string()
        )


show(
    "YEARLY SOURCE AUDIT",
    YEAR_AUDIT
)


show(
    "YEAR BOUNDARY AUDIT",
    BOUNDARY_AUDIT
)


show(
    "CONTIGUOUS >5% JUMPS",
    EXTREME_5
)


show(
    "IMMEDIATE >5% REVERSALS",
    REVERSALS
)


show(
    "REAL / POSSIBLE LARGE DAILY MOVES",
    LARGE_DAILY_SPAN
)


show(
    "AGGREGATE FILE vs YEARLY FILE DIFFERENCES",
    AGGREGATE_COMPARISON,
    50
)


show(
    "FINAL REBUILD SUMMARY",
    SUMMARY
)


print()
print("=" * 105)
print("FINAL DECISION")
print("=" * 105)

print(
    FINAL_DECISION
)


if hard_failures:

    print()

    for failure in hard_failures:

        print(
            "-",
            failure
        )


else:

    print()
    print(
        "年別15分CSVだけから、一貫したhistorical datasetを再構築できました。"
    )

    print(
        "次のステップでこれを正式な `bars` に切り替え、"
        "全バックテストを最初から再計算できます。"
    )


# ============================================================
# 16. Clean bars作成
# ============================================================

bars_rebuilt_clean = REBUILT[
    [
        "open",
        "high",
        "low",
        "close",
    ]
].copy()


# ============================================================
# 17. CSV保存
# ============================================================

clean_path = (

    OUTPUT_DIR

    /

    "usdjpy_15m_clean_yearly_2016_2026.csv"
)


bars_rebuilt_clean.to_csv(
    clean_path,
    index_label="timestamp"
)


YEAR_AUDIT.to_csv(
    OUTPUT_DIR /
    "yearly_source_audit.csv",
    index=False
)


BOUNDARY_AUDIT.to_csv(
    OUTPUT_DIR /
    "year_boundary_audit.csv",
    index=False
)


SUMMARY.to_csv(
    OUTPUT_DIR /
    "rebuild_summary.csv",
    index=False
)


AGGREGATE_COMPARISON.to_csv(
    OUTPUT_DIR /
    "aggregate_vs_yearly_differences.csv",
    index_label="timestamp"
)


# Notebook variables

CLEAN_REBUILD_YEARLY_AUDIT = (
    YEAR_AUDIT.copy()
)

CLEAN_REBUILD_BOUNDARIES = (
    BOUNDARY_AUDIT.copy()
)

CLEAN_REBUILD_SUMMARY = (
    SUMMARY.copy()
)

CLEAN_REBUILD_AGG_DIFF = (
    AGGREGATE_COMPARISON.copy()
)

CLEAN_REBUILD_DECISION = (
    FINAL_DECISION
)


print()
print("=" * 105)
print("REBUILD COMPLETE")
print("=" * 105)

print(
    "bars_rebuilt_clean:",
    f"{len(bars_rebuilt_clean):,}",
    "rows"
)

print(
    "Saved:",
    clean_path
)


## 元セルindex 65
構文状態：valid


In [ ]:
# ============================================================
# PROMOTE CLEAN DATASET
# clean yearly datasetを正式なbarsへ切り替える
# ============================================================

import pandas as pd
import numpy as np

if "bars_rebuilt_clean" not in globals():
    raise RuntimeError(
        "bars_rebuilt_clean がありません。"
        "clean dataset rebuildセルを先に実行してください。"
    )

# 旧barsを退避
if "bars" in globals():
    bars_before_clean_rebuild = bars.copy()

# 正式切替
bars = bars_rebuilt_clean.copy()

bars = bars.sort_index()

# ------------------------------------------------------------
# 最終 sanity check
# ------------------------------------------------------------

if not isinstance(bars.index, pd.DatetimeIndex):
    raise RuntimeError("bars.index がDatetimeIndexではありません。")

if bars.index.tz is None:
    bars.index = bars.index.tz_localize("UTC")
else:
    bars.index = bars.index.tz_convert("UTC")

if bars.index.duplicated().any():
    raise RuntimeError("duplicate timestampsがあります。")

bad_grid = (
    (bars.index.minute % 15 != 0)
    |
    (bars.index.second != 0)
    |
    (bars.index.microsecond != 0)
)

if np.asarray(bad_grid).any():
    raise RuntimeError("15分grid外timestampがあります。")

required = ["open", "high", "low", "close"]

missing = [
    c for c in required
    if c not in bars.columns
]

if missing:
    raise RuntimeError(
        f"OHLC列不足: {missing}"
    )

print("=" * 80)
print("CLEAN DATASET PROMOTED")
print("=" * 80)

print(
    "Rows:",
    f"{len(bars):,}"
)

print(
    "Period:",
    bars.index.min(),
    "->",
    bars.index.max()
)

print(
    "Duplicate timestamps:",
    int(bars.index.duplicated().sum())
)

print(
    "Bad 15m timestamps:",
    int(np.asarray(bad_grid).sum())
)

print()
print(
    "正式な bars を clean yearly dataset に切り替えました。"
)

print()
print(
    "次: HGB BASE FULL PIPELINEをclean dataで"
    "2020-2026まで再検証する。"
)
